# DataMate — Qwen2.5-Coder-7B-Instruct · 208 mẫu · Kaggle T4 ×2

Train **plain FP16 LoRA + FSDP1**, không lượng tử hóa. Dataset dùng prompt folder + README + schema/dtype + câu hỏi/ngày mốc.
Giữ đủ 208 mẫu, bao gồm hội thoại nhiều lượt; loss chỉ ở assistant cuối. Không sinh thêm dataset, không chạy code doanh nghiệp và không chấm PA trong notebook này.

**Chuẩn bị:** chọn **GPU T4 ×2**, bật Internet để cài thư viện; Add Input gói `datamate_qwen7b_208_kaggle_input.zip` dưới dạng **private Dataset** (Kaggle giải nén khi tạo Dataset), và Add Input weights **Qwen2.5-Coder-7B-Instruct**.
Weights 7B không nằm trong ZIP nhỏ. Không cần raw data/gold hoặc Google Drive.

Đã kiểm tra tokenizer thật: **208 mẫu, 1.086–6.766 token/toàn hội thoại**. Chưa chạy xác minh trên Kaggle 2×T4; cell smoke sẽ kiểm tra bộ nhớ, export và restore trước lượt train đầy đủ.

## 1. Cấu hình phiên chạy

In [ ]:
from pathlib import Path
import os, json, hashlib, shutil, subprocess, sys, math

MODEL_DIR = ''  # Để trống để tự tìm; nếu nhiều model thì nhập đường dẫn Instruct trong /kaggle/input.
RUN_NAME = 'qwen7b_208_fp16_lora'
MAX_SEQ_LEN = 8192
EPOCHS = 3.0
LEARNING_RATE = 1e-4
GRAD_ACCUM = 4  # batch 1/GPU × 2 GPU × 4 = 8 mẫu/optimizer step.
RESUME_FROM_CHECKPOINT = ''  # 'latest' hoặc đường dẫn checkpoint-N từ Output phiên cũ.
HF_REPO_NAME = 'Zalo-challenge'  # Repo private dưới tài khoản HF_TOKEN; để '' nếu chỉ cần log local.

EXPECTED_DATA_SHA256 = 'cfa70a9af1b87d282ff5428c7bf7afd9a3308a449a2095d23b320479284fd8fc'
WORK = Path('/kaggle/working/datamate_qwen7b_208')
RUN_ROOT = WORK / 'runs' / RUN_NAME
WORK.mkdir(parents=True, exist_ok=True)
os.environ.update(USE_TF='0', USE_TORCH='1', TOKENIZERS_PARALLELISM='false',
                  NCCL_P2P_DISABLE='1', NCCL_IB_DISABLE='1', OMP_NUM_THREADS='2', PYTHONUNBUFFERED='1')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Phiên này chưa có NVIDIA GPU. Chọn Settings → Accelerator → GPU T4 x2 trước Save & Run All.')
subprocess.run(['nvidia-smi'], check=True)

## 2. Tìm và kiểm tra gói source + dataset 208 mẫu

In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

candidates = []
for candidate in Path('/kaggle/input').rglob('bundle_manifest.json'):
    manifest = json.loads(candidate.read_text(encoding='utf-8'))
    if manifest.get('bundle') == 'datamate-qwen7b-sft208-2t4':
        candidates.append((candidate.parent, manifest))
if len(candidates) != 1:
    raise RuntimeError(f'Cần đúng 1 Input bundle SFT208 đã giải nén, tìm được {len(candidates)}. Kiểm tra Add Input.')
bundle, manifest = candidates[0]
assert manifest.get('revision') == 'hub-backup-v1', 'Hãy cập nhật Input Dataset bằng ZIP mới có HF backup.'
for relative, expected in manifest['files'].items():
    assert sha256(bundle / relative) == expected, f'Bundle checksum mismatch: {relative}'
assert manifest['dataset_sha256'] == EXPECTED_DATA_SHA256
PROJECT = WORK / 'source'
shutil.copytree(bundle, PROJECT, dirs_exist_ok=True)
DATASET = PROJECT / 'dataset/sft_train_ready.jsonl'
assert sha256(DATASET) == EXPECTED_DATA_SHA256
records = [json.loads(line) for line in DATASET.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(records) == 208 and len({row['id'] for row in records}) == 208
assert {row['prompt_format'] for row in records} == {'folder_readme_schema_v1'}
os.chdir(PROJECT)
print('Source:', PROJECT)
print('Dataset:', DATASET, '| records:', len(records))

## 3. Cài đúng phiên bản thư viện, giữ CUDA PyTorch của Kaggle

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle-sft.txt'], check=True)
# Optional torchao có thể được cài sẵn với version không tương thích PEFT; LoRA FP16 không dùng nó.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', """
import torch, transformers, peft, accelerate
from packaging.version import Version
from torch import nn
from peft import LoraConfig, get_peft_model
assert Version(torch.__version__.split('+')[0]) >= Version('2.6')
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, 'Chọn T4 x2 trong Settings'
assert all(torch.cuda.get_device_capability(i) == (7,5) for i in range(2))
model = get_peft_model(nn.Sequential(nn.Linear(4,4)), LoraConfig(target_modules=['0'], r=2))
assert any('lora_' in n and p.requires_grad for n,p in model.named_parameters())
print('torch',torch.__version__,'transformers',transformers.__version__,'peft',peft.__version__,'accelerate',accelerate.__version__)
for i in range(2): print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/2**30,2),'GiB')
"""], check=True)

## 4. Backup tự động lên Hugging Face private

Trong Kaggle: **Add-ons → Secrets**, thêm `HF_TOKEN` có quyền write và bật quyền truy cập secret cho notebook này.
Không đặt token vào cell hoặc log. Cell tự lấy username từ token rồi tạo repo private `username/Zalo-challenge`.
Checkpoint được snapshot và upload từ rank 0 trong thread nền; train tiếp tục, không đợi network mỗi lần save.
Nếu upload lỗi, log có `[HF backup] upload_failed` và bản local được giữ lại. Để chỉ xem log, đặt `HF_REPO_NAME=''` ở cell 1.

In [ ]:
if HF_REPO_NAME:
    from datetime import datetime, timezone
    os.environ['HF_HUB_OFFLINE'] = '0'
    from huggingface_hub import HfApi
    token = os.environ.get('HF_TOKEN')
    if not token:
        from kaggle_secrets import UserSecretsClient
        try:
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            raise RuntimeError('Thêm HF_TOKEN có quyền write trong Add-ons → Secrets và bật secret cho notebook. Hoặc đặt HF_REPO_NAME="" để chỉ xem log.') from None
    os.environ['HF_TOKEN'] = token
    api = HfApi(token=token)
    repo_id = HF_REPO_NAME if '/' in HF_REPO_NAME else api.whoami()['name'] + '/' + HF_REPO_NAME
    api.create_repo(repo_id=repo_id, repo_type='model', private=True, exist_ok=True)
    assert api.repo_info(repo_id, repo_type='model').private, 'Chỉ dùng repo private; repo hiện tại đang public.'
    os.environ['DATAMATE_HF_REPO'] = repo_id
    os.environ['DATAMATE_HF_PREFIX'] = 'runs/' + RUN_NAME + '/' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    print('Backup repo:', 'https://huggingface.co/' + repo_id)
else:
    os.environ.pop('DATAMATE_HF_REPO', None)
    os.environ.pop('DATAMATE_HF_PREFIX', None)
    print('HF backup tắt; log vẫn được stream và lưu trong Kaggle Output.')

## 5. Chọn weights Instruct cục bộ và kiểm tra đủ shard

In [ ]:
def is_qwen7b(path):
    try:
        cfg = json.loads((path / 'config.json').read_text(encoding='utf-8'))
        return (cfg.get('model_type') == 'qwen2' and cfg.get('hidden_size') == 3584
                and cfg.get('num_hidden_layers') == 28 and (path / 'tokenizer_config.json').is_file())
    except (OSError, ValueError):
        return False

if MODEL_DIR:
    model_path = Path(MODEL_DIR)
else:
    models = sorted({p.parent for p in Path('/kaggle/input').rglob('config.json') if is_qwen7b(p.parent)})
    if len(models) != 1:
        raise RuntimeError(f'Tìm được {len(models)} model Qwen7B: {models}. Điền MODEL_DIR ở cell 1.')
    model_path = models[0]
assert is_qwen7b(model_path), f'Không phải cấu hình Qwen7B hợp lệ: {model_path}'
index = model_path / 'model.safetensors.index.json'
if index.is_file():
    required = set(json.loads(index.read_text())['weight_map'].values())
else:
    required = {'model.safetensors'}
missing = [name for name in sorted(required) if not (model_path / name).is_file()]
assert not missing, f'Thiếu weights: {missing}'
MODEL_DIR = str(model_path.resolve())
print('Model:', MODEL_DIR)
print('Xác nhận Input này là Qwen2.5-Coder-7B-Instruct; cấu hình kiến trúc không chứng minh bản Instruct.')

## 6. Kiểm tra token thật — không nạp weights, không cắt/bỏ mẫu

In [ ]:
from collections import deque
import signal

def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=40)
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1,
                              start_new_session=True) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line); log.flush(); tail.append(line)
                status = process.wait()
            except KeyboardInterrupt:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait()
                raise
    if status:
        raise RuntimeError(f'Lệnh lỗi (exit {status}). Log: {log_path}\n' + ''.join(tail))

BASE_ARGS = ['--model', MODEL_DIR, '--data', str(DATASET), '--max-seq-len', str(MAX_SEQ_LEN),
             '--epochs', str(EPOCHS), '--lr', str(LEARNING_RATE), '--grad-accum', str(GRAD_ACCUM)]
run_logged([sys.executable, '-u', 'scripts/train_sft_kaggle_2t4.py', *BASE_ARGS,
            '--output-dir', str(RUN_ROOT / 'preflight'), '--prepare-only'], RUN_ROOT / 'preflight.log')
report = json.loads((RUN_ROOT / 'preflight/dataset_report.json').read_text())
assert report['records'] == 208 and not report['overlength']
print('Đủ 208 mẫu. Tokens:', report['min_tokens'], '→', report['max_tokens'])

## 7. Smoke trên hai T4 — 2 optimizer steps với mẫu dài nhất

Cell này train thử, lưu adapter/optimizer rồi khôi phục checkpoint ngay trong phiên để kiểm tra đường save/load.
Nếu OOM hoặc lỗi thì dừng ở đây; chưa chạy full. Khi chạy lại smoke, xóa marker cũ bằng cách dùng RUN_NAME mới.

In [ ]:
LAUNCH = [sys.executable, '-m', 'accelerate.commands.launch', '--config_file',
          'configs/accelerate_kaggle_2t4_sft.yaml', 'scripts/train_sft_kaggle_2t4.py']
run_logged([*LAUNCH, *BASE_ARGS, '--output-dir', str(RUN_ROOT / 'smoke'), '--smoke'], RUN_ROOT / 'smoke.log')
smoke = json.loads((RUN_ROOT / 'smoke/completed.json').read_text())
assert smoke['status'] == 'passed' and smoke['checkpoint_restore_checked']
assert {r['rank'] for r in smoke['ranks']} == {0, 1}
assert all(r['steps'] == 2 and math.isfinite(r['loss']) and r['peak_vram_gib'] > 0 for r in smoke['ranks'])
print(json.dumps(smoke['ranks'], indent=2))

## 8. Train đủ 208 mẫu

Mặc định 3 epochs, LR 1e-4, LoRA r16/alpha32, effective batch 8. Đây là cấu hình khởi đầu, chưa phải hyperparameters được benchmark tối ưu.
Checkpoint mỗi 10 optimizer steps, giữ 2 bản và luôn lưu checkpoint cuối. Chỉ dùng checkpoint mình tạo.

Resume cùng phiên: đặt `RESUME_FROM_CHECKPOINT='latest'`. Phiên mới: gắn Output cũ thành Input và nhập đường dẫn `checkpoint-N` đầy đủ. Giữ nguyên model Input/path, data, settings và versions.

In [ ]:
assert (RUN_ROOT / 'smoke/completed.json').is_file(), 'Chạy smoke trước'
command = [*LAUNCH, *BASE_ARGS, '--output-dir', str(RUN_ROOT / 'train')]
if RESUME_FROM_CHECKPOINT:
    command += ['--resume', RESUME_FROM_CHECKPOINT]
run_logged(command, RUN_ROOT / 'train.log')
completed = json.loads((RUN_ROOT / 'train/completed.json').read_text())
assert completed['status'] == 'passed'
print('Adapter:', RUN_ROOT / 'train/final_adapter')
print(json.dumps(completed['ranks'], indent=2))

## 9. Đóng gói và lưu kết quả

Adapter PEFT dùng lại được trong pipeline inference với base Qwen7B. ZIP gồm adapter, checkpoint resume, log, metadata và dataset 208 mẫu.
Chọn **Save Version → Save & Run All** để Kaggle lưu Output khi phiên chạy hoàn tất. Với interactive session, tải ZIP hoặc lưu Output trước khi kết thúc máy ảo.

Notebook này không đo PA. Pilot 30 câu hiện có 4 câu trùng bộ SFT nên không dùng toàn bộ pilot đó để kết luận khả năng trả lời câu chưa học.

In [ ]:
assert (RUN_ROOT / 'train/final_adapter/adapter_model.safetensors').is_file()
assert (RUN_ROOT / 'train/final_adapter/adapter_config.json').is_file()
shutil.copy2(DATASET, RUN_ROOT / 'sft_train_ready.jsonl')
archive = shutil.make_archive(str(Path('/kaggle/working') / (RUN_NAME + '_results')), 'zip', RUN_ROOT)
print('ZIP kết quả:', archive)
from IPython.display import FileLink, display
display(FileLink(archive))